In [ ]:
#Installs
!pip install umap-learn hdbscan nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 3.1 MB/s eta 0:00:00


In [ ]:
#IMPORTS
import pandas as pd
import numpy as np
import os
import glob
from gensim.models import Word2Vec
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
import hdbscan
import umap.umap_ as umap
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
import plotly.express as px

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from google.colab import drive
drive.mount("/content/drive") #Datasets were uploaded to Google Drive and fetched from there

Mounted at /content/drive


In [ ]:
def create_dataframe(directory):
    causes = []
    effects = []

    #Glob to get all CSV files in the directory
    csv_files = glob.glob(os.path.join(directory, '*.csv'))

    for file in csv_files:
        try:
            df = pd.read_csv(file)

            #Check if "cause" and "effect" columns exist (lowercase)
            if 'cause' in df.columns and 'effect' in df.columns:
                # Replace missing values with 'NA' and convert to strings
                df['cause'] = df['cause'].fillna('NA').astype(str)
                df['effect'] = df['effect'].fillna('NA').astype(str)

                #Append the "cause" and "effect" data to the lists
                causes.extend(df['cause'].tolist())
                effects.extend(df['effect'].tolist())
            else:
                print(f"Skipping file {file} as it doesn't contain 'cause' and 'effect' columns")
        except Exception as e:
            print(f"Error reading file {file}: {e}")

    #Create a Dataframe from the causes and effects with columns "Cause" and "Effect"
    data = pd.DataFrame({'Cause': causes, 'Effect': effects})
    return data

data = create_dataframe('/content/drive/MyDrive/extracted_causes_effects/ThirdDataset_PDF_cause-effect') #Change dataset if needed

#Inspect dataset if needed:
# data.head(50)

# data["Cause"].str.len()

# data["Effect"].str.len().mean()

,Cause,Effect
0,##words : Ostracism Social exclusion Psychosoc...,Gender differences
1,disturbances.,","
2,has been shown both to relate to psychosocial ...,hypothalamus pituitary ad reno cortical axis (...
3,##c,does not affect psychological responses to pub...
4,- women,a blunted cortisol stress response
5,- experience of social exclusion,blunted cortisol response to stress in women b...
6,factor might,higher vulnerability to social triggers of hea...
7,ameliorates the psychological impact of stress...,support has several protective effects
8,support,##k of
9,disturbances,","


In [ ]:
#Tokenize the Cause and Effect columns
data['Cause_Tokens'] = data['Cause'].apply(lambda x: nltk.word_tokenize(x.lower()))

data['Effect_Tokens'] = data['Effect'].apply(lambda x: nltk.word_tokenize(x.lower()))

In [ ]:
#Add stop words and add common punctuation marks to them since there are many unnecessary marks in the causes and effects
stop_words = set(stopwords.words('english'))
stop_words.update({'#', ',', '.', '-', '?'})

#Remove stop words from Cause_Tokens and Effect_Tokens
data['Cause_Tokens'] = data['Cause_Tokens'].apply(lambda tokens: [t for t in tokens if t not in stop_words])

data['Effect_Tokens'] = data['Effect_Tokens'].apply(lambda tokens: [t for t in tokens if t not in stop_words])

In [ ]:
#Prepare tokenized sentences separately for causes and effects
cause_sentences = data['Cause_Tokens'].tolist()
effect_sentences = data['Effect_Tokens'].tolist()

In [ ]:
#Train Word2Vec model for causes and effects
cause_model = Word2Vec(cause_sentences, vector_size=100, window=5, min_count=1, workers=4)

effect_model = Word2Vec(effect_sentences, vector_size=100, window=5, min_count=1, workers=4)

In [ ]:
#Defining function for averaging all word embeddings in a sentence
#Taken from https://ai.intelligentonlinetools.com/ml/text-clustering-word-embedding-machine-learning/

def sent_vectorizer(sent, model):
    sent_vec = np.zeros(model.vector_size)
    numw = 0
    for w in sent:
        if w in model.wv:
            sent_vec += model.wv[w]
            numw += 1
    if numw > 0:
        sent_vec = sent_vec / numw
    return sent_vec

In [ ]:
#Generate embeddings for Cause_Tokens and Effect_Tokens
data['Cause_Embedding'] = data['Cause_Tokens'].apply(lambda x: sent_vectorizer(x, cause_model))

data['Effect_Embedding'] = data['Effect_Tokens'].apply(lambda x: sent_vectorizer(x, effect_model))

#Convert embeddings to numpy arrays
cause_embeddings = np.vstack(data['Cause_Embedding'].values)
effect_embeddings = np.vstack(data['Effect_Embedding'].values)

In [ ]:
%%time
from umap import UMAP

# Initialize UMAP model
umap_model = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine')

# Reduce dimensions for causes and effects
reduced_cause_embeddings = umap_model.fit_transform(cause_embeddings)
reduced_effect_embeddings = umap_model.fit_transform(effect_embeddings)

/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


CPU times: user 1min 9s, sys: 655 ms, total: 1min 9s
Wall time: 56.8 s


In [ ]:
import hdbscan

#HDBSCAN parameters
hdbscan_params = {
    'min_cluster_size': 10,
    'metric': 'euclidean'
}

#Cluster causes and effects
cause_clusterer = hdbscan.HDBSCAN(**hdbscan_params)
cause_labels = cause_clusterer.fit_predict(reduced_cause_embeddings)

effect_clusterer = hdbscan.HDBSCAN(**hdbscan_params)
effect_labels = effect_clusterer.fit_predict(reduced_effect_embeddings)

/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
#Print the number of clusters
print(np.unique(effect_labels))

[ -1   0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16
  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34
  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52
  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70
  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88
  89  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106
 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124
 125 126 127 128]


In [ ]:
#Prepare DataFrames for causes and effects
data_vis_causes = pd.DataFrame({
    'UMAP1': reduced_cause_embeddings[:, 0],
    'UMAP2': reduced_cause_embeddings[:, 1],
    'Cluster': cause_labels,
    'Text': data['Cause']
})

data_vis_effects = pd.DataFrame({
    'UMAP1': reduced_effect_embeddings[:, 0],
    'UMAP2': reduced_effect_embeddings[:, 1],
    'Cluster': effect_labels,
    'Text': data['Effect']
})

In [ ]:
#Visualization
#Font sizes have been increased to show the labels better in the thesis

#Plot causes
fig_causes = px.scatter(
    data_vis_causes[data_vis_causes['Cluster'] != -1],
    x='UMAP1',
    y='UMAP2',
    color='Cluster',
    hover_data={'Text': True, 'Cluster': True},
    title="HDBSCAN Clusters for Causes in Reduced UMAP Space",
    labels={'UMAP1': 'UMAP Dimension 1', 'UMAP2': 'UMAP Dimension 2'},
    color_discrete_sequence=px.colors.qualitative.Vivid
)
fig_causes.update_traces(marker=dict(size=8, opacity=0.7))
fig_causes.update_layout(
    title_font_size=16,
    legend_title="Cluster Label",
    margin=dict(l=40, r=40, t=40, b=40),
    xaxis_title_font=dict(size=25, family = "Times New Roman"),  #x-axis title font size
    yaxis_title_font=dict(size=25, family = "Times New Roman"),  #y-axis title font size
    xaxis_tickfont=dict(size=20, family = "Times New Roman"),    #x-axis tick label size
    yaxis_tickfont=dict(size=20, family = "Times New Roman"),    #y-axis tick label size
    coloraxis_colorbar=dict(
        tickfont=dict(size=20, family = "Times New Roman"),      #Font size of color bar numbers
        title_font=dict(size=25, family = "Times New Roman")    #Font size of color bar title
    )
)
fig_causes.show()

#Plot effects
fig_effects = px.scatter(
    data_vis_effects[data_vis_effects['Cluster'] != -1],  # Exclude noise
    x='UMAP1',
    y='UMAP2',
    color='Cluster',
    hover_data={'Text': True, 'Cluster': True},
    title="HDBSCAN Clusters for Effects in Reduced UMAP Space",
    labels={'UMAP1': 'UMAP Dimension 1', 'UMAP2': 'UMAP Dimension 2'},
    color_discrete_sequence=px.colors.qualitative.Pastel1
)
fig_effects.update_traces(marker=dict(size=8, opacity=0.7))
fig_effects.update_layout(
    title_font_size=16,
    legend_title="Cluster Label",
    margin=dict(l=40, r=40, t=40, b=40),
    xaxis_title_font=dict(size=25, family = "Times New Roman"),  #x-axis title font size
    yaxis_title_font=dict(size=25, family = "Times New Roman"),  #y-axis title font size
    xaxis_tickfont=dict(size=20, family = "Times New Roman"),    #x-axis tick label size
    yaxis_tickfont=dict(size=20, family = "Times New Roman"),    #y-axis tick label size
    coloraxis_colorbar=dict(
        tickfont=dict(size=20, family = "Times New Roman"),      #Font size of color bar numbers
        title_font=dict(size=25, family = "Times New Roman")    #Font size of color bar title
    )
)
fig_effects.show()

In [ ]:
from sklearn.metrics import davies_bouldin_score, silhouette_score, calinski_harabasz_score

#METRICS

#Evaluation for causes
print("Evaluation Metrics for Causes Clustering:")
cause_labels_filtered = cause_labels[cause_labels != -1]  #Exclude noise points
cause_embeddings_filtered = reduced_cause_embeddings[cause_labels != -1]  #Exclude noise embeddings

if len(set(cause_labels_filtered)) > 1:  #Make sure there is more than one cluster existing
    db_score = davies_bouldin_score(cause_embeddings_filtered, cause_labels_filtered)
    silhouette = silhouette_score(cause_embeddings_filtered, cause_labels_filtered)
    ch_score = calinski_harabasz_score(cause_embeddings_filtered, cause_labels_filtered)

    print(f"Davies-Bouldin Index: {db_score:.3f} (Lower is better)")
    print(f"Silhouette Score: {silhouette:.3f} (Higher is better)")
    print(f"Calinski-Harabasz Score: {ch_score:.3f} (Higher is better)")
else:
    print("Not enough clusters to compute metrics")

#Evaluation for effects
print("\nEvaluation Metrics for Effects Clustering:")
effect_labels_filtered = effect_labels[effect_labels != -1]
effect_embeddings_filtered = reduced_effect_embeddings[effect_labels != -1]

if len(set(effect_labels_filtered)) > 1:
    db_score = davies_bouldin_score(effect_embeddings_filtered, effect_labels_filtered)
    silhouette = silhouette_score(effect_embeddings_filtered, effect_labels_filtered)
    ch_score = calinski_harabasz_score(effect_embeddings_filtered, effect_labels_filtered)

    print(f"Davies-Bouldin Index: {db_score:.3f} (Lower is better)")
    print(f"Silhouette Score: {silhouette:.3f} (Higher is better)")
    print(f"Calinski-Harabasz Score: {ch_score:.3f} (Higher is better)")
else:
    print("Not enough clusters to compute metrics")


Evaluation Metrics for Causes Clustering:
Davies-Bouldin Index: 0.361 (Lower is better)
Silhouette Score: 0.657 (Higher is better)
Calinski-Harabasz Score: 86138.507 (Higher is better)
